# Paired protocol: CIFAR-10 (Tables 1, 2 and 3 (ablation))

DA351 Phase 1, FVLC replication (Lai et al., ICLR 2024). This is the notebook we actually ran on **Kaggle** (GPU T4), with its outputs.

The repo was attached as a Kaggle dataset (Add Input). The results zip it produced is `results_paired/cifar10/` in this repo.
Only Kaggle's default boilerplate cell was removed, and pip progress bars were trimmed from the output; nothing else was changed.


In [2]:
import os, glob, shutil
os.makedirs('/tmp/code', exist_ok=True)
z = glob.glob('/kaggle/input/**/*fvlc-replication*.zip', recursive=True)
if z:
    !unzip -q -o "{z[0]}" -d /tmp/code/
else:
    src = os.path.dirname(glob.glob('/kaggle/input/**/run_experiments.py', recursive=True)[0])
    shutil.copytree(src, '/tmp/code/fvlc-replication', dirs_exist_ok=True)
%cd /tmp/code/fvlc-replication

!pip install -q ftfy regex tqdm nltk "pytorchcv>=0.0.67"
!pip install -q git+https://github.com/openai/CLIP.git
!python -c "import clip, pytorchcv, nltk; print('PACKAGES OK')"

# CIFAR: reuse a Kaggle copy if one is attached (avoids the slow 40-min download)
os.environ['FVLC_DATA_ROOT'] = '/tmp/data'
os.makedirs('/tmp/data', exist_ok=True)
for name in ['cifar-10-batches-py', 'cifar-100-python']:
    hit = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if hit and not os.path.exists(f'/tmp/data/{name}'):
        shutil.copytree(hit[0], f'/tmp/data/{name}')
        print('Using Kaggle copy of', name)

os.environ['FVLC_ACT_DIR'] = '/tmp/activations'
OUT = '/kaggle/working/results_paired'
!grep -c "def evaluate_paired" fvlc/pipeline.py
!python tests/test_core.py

/tmp/code/fvlc-replication
  Preparing metadata (setup.py) ... done
PACKAGES OK
1
100%|███████████████████████████████████████████| 50/50 [00:00<00:00, 97.59it/s]
did not converge at 50 iterations (criteria False)
(0) lambda 0.0007, loss 0.0270, acc 1.0000 [val acc 1.0000] [test acc -1.0000], sparsity 0.9791666666666666 [47/48], time 0.529217004776001, lr 0.1000
universal: TCPC base 1.338e-01 -> FVLC 9.998e-03 | TOPC 5.930e-02 -> 4.852e-03
per_sample: TCPC base 1.338e-01 -> FVLC 1.890e-02 | TOPC 5.930e-02 -> 8.786e-03
all tests passed


In [3]:
!python run_experiments.py --dataset cifar10 --protocol paired --fvlc_init gaussian --ablation --ablation_perturbations WP1_10 IP WP1_10+WP2+IP --out $OUT
!python make_tables.py --results $OUT

Dataset cifar10 | device cuda | seed 42
100%|█████████████████████████████████████████| 170M/170M [25:08<00:00, 113kB/s]
100%|███████████████████████████████████████| 244M/244M [00:04<00:00, 51.6MiB/s]
100%|████████████████████████████████████████| 335M/335M [00:02<00:00, 125MiB/s]
[vanilla] 143 concepts -> 138 after CLIP cutoff                                 
  projection: best step 200, val similarity 0.6065
[vanilla] 131 concepts after interpretability cutoff
  8%|███▍                                     | 85/1000 [00:46<08:17,  1.84it/s]
(0) lambda 0.0007, loss 0.4244, acc 0.8697 [val acc 0.8651] [test acc -1.0000], sparsity 0.2679389312977099 [351/1310], time 46.67749857902527, lr 0.1000
[vanilla] test acc 0.8651  (351/1310 non-zero final weights)
  6%|██▍                                      | 59/1000 [00:37<09:56,  1.58it/s]
(0) lambda 0.0007, loss 0.4027, acc 0.8962 [val acc 0.8831] [test acc -1.0000], sparsity 0.178515625 [1828/10240], time 37.92536473274231, lr 0.1000
[stand

In [4]:
!cd /kaggle/working && zip -qr results_cifar10.zip results_paired -x "*/layers/*" "*/vanilla.pt"
print(open(f'{OUT}/tables.md').read())

# Replication results vs. paper

Ours = this repo (seed 42, evaluation protocol: paired). Paper = Lai et al., ICLR 2024. Accuracy in %.


## Headline numbers (averaged over the 7 paper perturbations)

| Dataset | LF-CBM acc ours / paper | FVLC acc ours / paper | base TCPC ours / paper | FVLC TCPC ours / paper | base TOPC ours / paper | FVLC TOPC ours / paper | seed-only TCPC ours |
|---|---|---|---|---|---|---|---|
| CIFAR-10 | 86.51 / 86.32 | 86.50 / 86.48 | 2.88e-01 / 1.59e-01 | 8.36e-03 / 8.49e-03 | 1.58e-01 / 6.94e-02 | 5.11e-03 / 8.70e-03 | 1.74e-01 |

## Table 1 – Accuracy

| Method | CIFAR-10 ours | CIFAR-10 paper |
|---|---|---|
| Standard (no interpretability) | 88.31 | 88.80 |
| Label-free CBM | 86.51 | 86.32 |
| WP1(5%) – base | 85.19 | 86.47 |
| WP1(5%) – FVLC | 86.50 | 86.34 |
| WP1(10%) – base | 84.44 | 86.25 |
| WP1(10%) – FVLC | 86.50 | 86.39 |
| WP2 – base | 85.93 | 86.41 |
| WP2 – FVLC | 86.50 | 86.22 |
| IP – base | 86.25 | 86.62 |
| IP – FVLC | 86.50 | 86.88 |
| WP1